In [33]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
import tifffile as tiff
from sklearn.metrics import classification_report
import keras_tuner as kt

In [2]:
CSV_PATH = r"C:\Users\rosha\Downloads\ailearn\HyperLeaf2024\train.csv"
DATA_DIR = r"C:\Users\rosha\Downloads\ailearn\HyperLeaf2024\images"
NUM_BANDS = 204
NUM_CLASSES = 4


## PreProcessing ##

In [3]:

df = pd.read_csv(CSV_PATH)

# Convert text cultivars ('Heerup', 'Kvium', etc.) to integers (0, 1, 2, 3)
image_ids = df['ImageId'].values

# The dataset uses one-hot encoded columns for the 4 classes
cultivar_cols = ['Heerup', 'Kvium', 'Rembrandt', 'Sheriff']

# Extract those 4 columns as a NumPy matrix and use argmax to convert the 1s into class indices (0, 1, 2, or 3)
labels = np.argmax(df[cultivar_cols].values, axis=1)

# Initialize empty arrays to hold our compressed 1D signatures
NUM_STATS=5
X_data = np.zeros((len(image_ids), NUM_BANDS,NUM_STATS), dtype=np.float32)

y_data = labels.astype(np.int32)

print("Processing hyperspectral cubes into 1D spectral vectors...")

for i, img_id in enumerate(image_ids):
    # 1. Update the extension to .tiff or .tif based on your folder
    file_path = os.path.join(DATA_DIR, f"{img_id:05d}.tiff")

    # 2. Load the multi-band TIFF
    cube = tiff.imread(file_path)

    # 3. CRITICAL: Fix the shape if bands are the first dimension
    # If shape is (204, Height, Width), change it to (Height, Width, 204)
    if cube.shape[0] == NUM_BANDS:
        cube = np.transpose(cube, (1, 2, 0))

    # Step A: Background Masking
    # Calculate the mean int
    # ensity per pixel across all 204 bands
    mean_intensity = np.mean(cube, axis=-1)

    # Define a threshold to isolate the leaf from the dark background
    threshold = np.percentile(mean_intensity, 15)
    leaf_mask = mean_intensity > threshold

    # Step B: Isolate leaf pixels and flatten spatially
    leaf_pixels = cube[leaf_mask]

    # Step C: Spatial Pooling (Average the chemical signature)
    if leaf_pixels.shape[0] == 0:
        signature = np.zeros((NUM_BANDS,NUM_STATS),dtype=np.float32)
    else:
        mean_signature = np.mean(leaf_pixels, axis=0)
        std_signature = np.std(leaf_pixels, axis=0)
        median_signature = np.median(leaf_pixels, axis=0)
        p25_signature = np.percentile(leaf_pixels, 25, axis=0)
        p75_signature = np.percentile(leaf_pixels, 75, axis=0)

        signature = np.stack([
                    mean_signature,
                    std_signature,
                    median_signature,
                    p25_signature,
                    p75_signature
                ],axis=-1)



    # Store the 1D vector
    X_data[i] = signature

print(f"Successfully processed {X_data.shape[0]} TIFF samples.")

print("Compressing and saving data to disk...")

np.savez_compressed("hyperleaf_processed.npz", X=X_data, y=y_data)
print("Saved successfully!")



Processing hyperspectral cubes into 1D spectral vectors...
Successfully processed 1590 TIFF samples.
Compressing and saving data to disk...
Saved successfully!


## MODEL ##


In [ ]:
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU Detected and Ready: {gpus}\n")
    except RuntimeError as e:
        print(f"GPU configuration error: {e}\n")
else:
    print("No GPU found. Falling back to CPU execution.\n")

# ==========================================
# Shared Data Loading, Splitting, Normalizing
# ==========================================
loaded_data = np.load("hyperleaf_processed.npz")
X_data = loaded_data['X']
y_data = loaded_data['y']
NUM_STATS = X_data.shape[2]
CLASS_NAMES = ['Heerup', 'Kvium', 'Rembrandt', 'Sheriff']

print(f"Loaded data shape: {X_data.shape}")

X_train, X_temp, y_train, y_temp = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42, stratify=y_data
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

X_min = X_train.min()
X_max = X_train.max()

X_train = (X_train - X_min) / (X_max - X_min + 1e-8)
X_val = (X_val - X_min) / (X_max - X_min + 1e-8)
X_test = (X_test - X_min) / (X_max - X_min + 1e-8)

print(f"Training shapes:   {X_train.shape} (80%)")
print(f"Validation shapes: {X_val.shape} (10%)")
print(f"Local Test shapes: {X_test.shape} (10%)")

def compile_and_train(model, model_name, epochs=200, batch_size=40):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    model.summary()

    checkpoint_path = f"best_{model_name}.keras"
    checkpoint = tf.keras.callbacks.ModelCheckpoint(
        checkpoint_path,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    )
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=12,
        min_lr=1e-6,
        verbose=1
    )

    print(f"\nStarting training: {model_name}")
    with tf.device("/GPU:0"):
        history = model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=[checkpoint, reduce_lr]
        )

    best_val_accuracy = max(history.history["val_accuracy"])
    best_val_epoch = int(np.argmax(history.history["val_accuracy"]) + 1)
    print(f"Best validation accuracy: {best_val_accuracy * 100:.2f}% at epoch {best_val_epoch}")

    best_model = tf.keras.models.load_model(checkpoint_path)
    test_loss, test_accuracy = best_model.evaluate(X_test, y_test, verbose=0)
    print(f"\n{model_name} local test accuracy: {test_accuracy * 100:.2f}%")

    predictions_probs = best_model.predict(X_test, verbose=0)
    predicted_indices = np.argmax(predictions_probs, axis=1)
    print(classification_report(y_test, predicted_indices, target_names=CLASS_NAMES))

    return best_model, history
"""def compile_and_train(model, model_name, epochs=200, batch_size=40):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    model.summary()

    checkpoint_path = f"best_{model_name}.keras"
    checkpoint = tf.keras.callbacks.ModelCheckpoint(
        checkpoint_path,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    )
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=12,
        min_lr=1e-6,
        verbose=1
    )

    print(f"\nStarting training: {model_name}")
    with tf.device("/GPU:0"):
        history = model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=[checkpoint, reduce_lr]
        )

    best_val_accuracy = max(history.history["val_accuracy"])
    best_val_epoch = int(np.argmax(history.history["val_accuracy"]) + 1)
    print(f"Best validation accuracy: {best_val_accuracy * 100:.2f}% at epoch {best_val_epoch}")

    best_model = tf.keras.models.load_model(checkpoint_path)
    test_loss, test_accuracy = best_model.evaluate(X_test, y_test, verbose=0)
    print(f"\n{model_name} local test accuracy: {test_accuracy * 100:.2f}%")

    predictions_probs = best_model.predict(X_test, verbose=0)
    predicted_indices = np.argmax(predictions_probs, axis=1)
    print(classification_report(y_test, predicted_indices, target_names=CLASS_NAMES))

    return best_model, history"""

### CNN Alone


In [46]:
tf.keras.backend.clear_session()

inputs = layers.Input(shape=(NUM_BANDS, NUM_STATS))

x = layers.Conv1D(128, kernel_size=5, activation='relu', padding='same')(inputs)
x = layers.MaxPooling1D(2)(x)

x = layers.Conv1D(256, kernel_size=5, activation='relu', padding='same')(x)
x = layers.MaxPooling1D(2)(x)

x = layers.Conv1D(256, kernel_size=5, activation='relu', padding='same')(x)
x = layers.Flatten()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

cnn_model = models.Model(inputs=inputs, outputs=outputs)
cnn_model, cnn_history = compile_and_train(
    cnn_model,
    "cnn_alone"
)
model = cnn_model
history = cnn_history
#86.6

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 204, 5)]          0         
                                                                 
 conv1d (Conv1D)             (None, 204, 128)          3328      
                                                                 
 max_pooling1d (MaxPooling1D  (None, 102, 128)         0         
 )                                                               
                                                                 
 conv1d_1 (Conv1D)           (None, 102, 256)          164096    
                                                                 
 max_pooling1d_1 (MaxPooling  (None, 51, 256)          0         
 1D)                                                             
                                                                 
 conv1d_2 (Conv1D)           (None, 51, 256)           327936

### CNN + Attention + BiGRU


In [48]:
tf.keras.backend.clear_session()

inputs = layers.Input(shape=(NUM_BANDS, NUM_STATS))

x = layers.Conv1D(128, kernel_size=5, activation='relu', padding='same')(inputs)
x = layers.MaxPooling1D(2)(x)

x = layers.Conv1D(256, kernel_size=5, activation='relu', padding='same')(x)
x = layers.MaxPooling1D(2)(x)

x = layers.Conv1D(256, kernel_size=5, activation='relu', padding='same')(x)

attention_output = layers.MultiHeadAttention(
    num_heads=2,
    key_dim=32,
    dropout=0.05
)(x, x)
x = layers.Add()([x, attention_output])
x = layers.LayerNormalization()(x)

x = layers.Bidirectional(layers.GRU(32, return_sequences=False))(x)
x = layers.Dropout(0.1)(x)
x = layers.Dense(64, activation='relu')(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

cnn_attention_bigru_model = models.Model(inputs=inputs, outputs=outputs)
cnn_attention_bigru_model, cnn_attention_bigru_history = compile_and_train(
    cnn_attention_bigru_model,
    "cnn_attention_bigru"
)
model = cnn_attention_bigru_model
history = cnn_attention_bigru_history
#92.45


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 204, 5)]     0           []                               
                                                                                                  
 conv1d (Conv1D)                (None, 204, 128)     3328        ['input_1[0][0]']                
                                                                                                  
 max_pooling1d (MaxPooling1D)   (None, 102, 128)     0           ['conv1d[0][0]']                 
                                                                                                  
 conv1d_1 (Conv1D)              (None, 102, 256)     164096      ['max_pooling1d[0][0]']          
                                                                                              

### CNN + BiGRU


In [ ]:
tf.keras.backend.clear_session()

inputs = layers.Input(shape=(NUM_BANDS, NUM_STATS))

x = layers.Conv1D(128, kernel_size=5, activation='relu', padding='same')(inputs)
x = layers.MaxPooling1D(2)(x)

x = layers.Conv1D(256, kernel_size=5, activation='relu', padding='same')(x)
x = layers.MaxPooling1D(2)(x)

x = layers.Conv1D(256, kernel_size=5, activation='relu', padding='same')(x)
x = layers.Bidirectional(layers.GRU(32, return_sequences=False))(x)
x = layers.Dropout(0.1)(x)
x = layers.Dense(64, activation='relu')(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

cnn_bigru_model = models.Model(inputs=inputs, outputs=outputs)
cnn_bigru_model, cnn_bigru_history = compile_and_train(
    cnn_bigru_model,
    "cnn_bigru"
)
model = cnn_bigru_model
history = cnn_bigru_history
#83

### CNN + Attention


In [42]:
tf.keras.backend.clear_session()

inputs = layers.Input(shape=(NUM_BANDS, NUM_STATS))

x = layers.Conv1D(128, kernel_size=5, activation='relu', padding='same')(inputs)
x = layers.MaxPooling1D(2)(x)

x = layers.Conv1D(256, kernel_size=5, activation='relu', padding='same')(x)
x = layers.MaxPooling1D(2)(x)

x = layers.Conv1D(256, kernel_size=5, activation='relu', padding='same')(x)

attention_output = layers.MultiHeadAttention(
    num_heads=2,
    key_dim=32,
    dropout=0.05
)(x, x)
x = layers.Add()([x, attention_output])
x = layers.LayerNormalization()(x)

x = layers.Flatten()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

cnn_attention_model = models.Model(inputs=inputs, outputs=outputs)
cnn_attention_model, cnn_attention_history = compile_and_train(
    cnn_attention_model,
    "cnn_attention"
)
model = cnn_attention_model
history = cnn_attention_history
#89.31

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 204, 5)]     0           []                               
                                                                                                  
 conv1d (Conv1D)                (None, 204, 128)     3328        ['input_1[0][0]']                
                                                                                                  
 max_pooling1d (MaxPooling1D)   (None, 102, 128)     0           ['conv1d[0][0]']                 
                                                                                                  
 conv1d_1 (Conv1D)              (None, 102, 256)     164096      ['max_pooling1d[0][0]']          
                                                                                              

## CNN + MAMBA ##

In [44]:
class MambaBlock(layers.Layer):
    def __init__(self, d_model, d_state=32, d_conv=4, expand=2, **kwargs):
        super(MambaBlock, self).__init__(**kwargs)
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.expand = expand
        self.d_inner = int(self.expand * self.d_model)

    def build(self, input_shape):
        self.in_proj = layers.Dense(self.d_inner * 2, use_bias=False)
        self.conv1d = layers.Conv1D(
            filters=self.d_inner,
            kernel_size=self.d_conv,
            padding='same',
            groups=self.d_inner,
            activation='swish'
        )
        self.x_proj = layers.Dense(self.d_state * 2 + self.d_inner, use_bias=False)
        self.dt_proj = layers.Dense(self.d_inner, activation='softplus')
        self.out_proj = layers.Dense(self.d_model, use_bias=False)
        super(MambaBlock, self).build(input_shape)

    def call(self, x):
        # 1. Gated branch splitting
        projected = self.in_proj(x)
        x_branch, res_branch = tf.split(projected, num_or_size_splits=2, axis=-1)
        x_branch = self.conv1d(x_branch)

        # 2. Extract State Space parameters
        ssm_params = self.x_proj(x_branch)
        B, C, delta = tf.split(ssm_params, [self.d_state, self.d_state, self.d_inner], axis=-1)
        delta = self.dt_proj(delta)

        # 🌟 THE VECTORIZED FIX: No more sequential item loops 🌟
        # We compute the discretization factors globally using cumulative products
        # This replaces the iterative loop with parallel tensor math that fits perfectly on the GPU
        delta_x = delta * x_branch
        state_influence = tf.reduce_mean(B, axis=-1, keepdims=True)
        decay_factors = tf.exp(-delta * tf.abs(state_influence))

        # Cumulative scan parallelizes the recurrence sequence across CUDA cores instantly
        cum_decay = tf.math.cumprod(decay_factors, axis=1)
        outputs = tf.math.cumsum(delta_x * cum_decay, axis=1) / (cum_decay + 1e-8)

        # Final output projection
        outputs = outputs * tf.reduce_mean(C, axis=-1, keepdims=True)
        gated_output = outputs * tf.keras.activations.swish(res_branch)
        return self.out_proj(gated_output)

    def get_config(self):
        config = super(MambaBlock, self).get_config()
        config.update({
            "d_model": self.d_model,
            "d_state": self.d_state,
            "d_conv": self.d_conv,
            "expand": self.expand
        })
        return config
inputs = layers.Input(shape=(NUM_BANDS, NUM_STATS))

# Block 1: Initial Local Convolution Feature Aggregator
x = layers.Conv1D(128, kernel_size=5, activation='relu', padding='same')(inputs)
x = layers.MaxPooling1D(2)(x)
x = layers.LayerNormalization()(x)

# Block 2: The Mamba State-Space Model Layer
# d_model matches the 64 incoming feature channels from the Conv1D block
mamba_output = MambaBlock(d_model=128, d_state=16, expand=2)(x)

# Residual Connection (Add original features back to the SSM outputs)
x = layers.Add()([x, mamba_output])
x = layers.LayerNormalization()(x)

# Block 3: Final Feature Refinement
x = layers.Conv1D(256, kernel_size=3, activation='relu', padding='same')(x)
x = layers.MaxPooling1D(2)(x)

# Flatten and Classify
x = layers.Flatten()(x)
x = layers.Dense(128, activation='relu')(x)
#x = layers.Dropout(0.1)(x) # Upgraded dropout to combat your training overfitting
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

# Define and Build Model
model = models.Model(inputs=inputs, outputs=outputs)

cnn_mamba_model = models.Model(inputs=inputs, outputs=outputs)
cnn_mamba_model, cnn_mamba_history = compile_and_train(
    cnn_mamba_model,
    "cnn_mamba"
)
model = cnn_mamba_model
history = cnn_mamba_history

#89.31

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_2 (InputLayer)           [(None, 204, 5)]     0           []                               
                                                                                                  
 conv1d_3 (Conv1D)              (None, 204, 128)     3328        ['input_2[0][0]']                
                                                                                                  
 max_pooling1d_2 (MaxPooling1D)  (None, 102, 128)    0           ['conv1d_3[0][0]']               
                                                                                                  
 layer_normalization_1 (LayerNo  (None, 102, 128)    256         ['max_pooling1d_2[0][0]']        
 rmalization)                                                                               

## TEST FROM TRAIN ##

In [ ]:
"""def build_tunable_model(hp):
    inputs = layers.Input(shape=(NUM_BANDS, NUM_STATS))

    # 1. Tune Initial Conv Block Filters
    conv1_filters = hp.Choice('conv1_filters', values=[32, 64, 128,256])
    x = layers.Conv1D(
        filters=conv1_filters,
        kernel_size=hp.Choice('conv1_kernel', values=[3, 5,7]),
        activation='relu',
        padding='same'
    )(inputs)
    x = layers.MaxPooling1D(2)(x)
    x = layers.LayerNormalization()(x)

    # 2. Tune Mamba Internal Latent Variables
    # We match d_model to the exact number of channels coming out of Conv1D
    mamba_state = hp.Int('mamba_d_state', min_value=8, max_value=32, step=8)
    mamba_expand = hp.Choice('mamba_expand', values=[1, 2, 4])

    mamba_output = MambaBlock(
        d_model=conv1_filters,
        d_state=mamba_state,
        expand=mamba_expand
    )(x)

    # Residual Connection
    x = layers.Add()([x, mamba_output])
    x = layers.LayerNormalization()(x)

    # 3. Tune Final Feature Refinement Block
    conv2_filters = hp.Choice('conv2_filters', values=[128, 256, 512])
    x = layers.Conv1D(filters=conv2_filters, kernel_size=3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)

    # 4. Dense Classification Head Tuning
    x = layers.Flatten()(x)

    dense_units = hp.Int('dense_units', min_value=64, max_value=256, step=64)
    x = layers.Dense(units=dense_units, activation='relu')(x)

    # Critical step to control overfitting dynamically
    dropout_rate = hp.Float('dropout_rate', min_value=0.1, max_value=0.5, step=0.1)
    x = layers.Dropout(dropout_rate)(x)

    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    # Compile with tunable learning rates
    model = models.Model(inputs=inputs, outputs=outputs)

    lr = hp.Choice('learning_rate', values=[0.1,0.01,1e-3, 5e-4, 1e-4])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# Create a dedicated tuner instance
tuner = kt.RandomSearch(
    build_tunable_model,
    objective='val_accuracy',
    max_trials=20,            # Total structural combinations to attempt
    executions_per_trial=1,   # How many times to retrain each model to verify performance
    directory='mamba_tuning', # Project folder name
    project_name='hyperleaf_ssm'
)

# Add early stopping to drop bad configurations early and save GPU time
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True
)

# Execute search using your optimized tf.data pipeline or raw splits
print("🚀 Starting Hyperparameter Optimization Loop...")
tuner.search(
    X_train,
    y_train,
    epochs=40, # 40 epochs is usually plenty to see if a combination is superior
    validation_data=(X_val, y_val),
    batch_size=128, # Keep batch size high to optimize GPU parallel calculations
    callbacks=[early_stop]
)

# 1. Retrieve the best parameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print("\n🏆 OPTIMIZATION COMPLETE. BEST PARAMETERS FOUND:")
print(f"-> Conv1 Filters: {best_hps.get('conv1_filters')}")
print(f"-> Mamba State Size (d_state): {best_hps.get('mamba_d_state')}")
print(f"-> Mamba Expansion Multiple: {best_hps.get('mamba_expand')}")
print(f"-> Dense Classification Units: {best_hps.get('dense_units')}")
print(f"-> Dropout Multiplier: {best_hps.get('dropout_rate')}")
print(f"-> Learning Rate Strategy: {best_hps.get('learning_rate')}")

# 2. Reconstruct the champion architecture
best_model = tuner.hypermodel.build(best_hps)

# 3. Train on full training constraints using your compile_and_train routine
final_mamba_model, history = compile_and_train(
    best_model,
    model_name="optimized_cnn_mamba",
    epochs=200,
    batch_size=128
)
#88.05"""

In [49]:
CLASS_NAMES = ['Heerup', 'Kvium', 'Rembrandt', 'Sheriff']

print("\n" + "="*60)
print("HOLDOUT TEST SET EVALUATION")
print("="*60)

predictions_probs = model.predict(X_test, verbose=0)
predicted_indices = np.argmax(predictions_probs, axis=1)

correct_count = 0
total_samples = len(y_test)

for i in range(total_samples):
    pred_idx = predicted_indices[i]
    true_idx = y_test[i]

    pred_name = CLASS_NAMES[pred_idx]
    true_name = CLASS_NAMES[true_idx]
    confidence = predictions_probs[i][pred_idx] * 100

    if pred_idx == true_idx:
        status = "CORRECT"
        correct_count += 1
    else:
        status = "WRONG"

    print(f"Sample #{i+1:03d} | Predicted: {pred_name: <10} ({confidence:.1f}%) | Actual: {true_name: <10} | {status}")

print("-" * 60)
final_accuracy = (correct_count / total_samples) * 100
print(f"HOLDOUT ACCURACY: {correct_count}/{total_samples} ({final_accuracy:.2f}%)")
print("-" * 60)

print("\nDETAILED CLASSIFICATION REPORT:")
print(classification_report(y_test, predicted_indices, target_names=CLASS_NAMES))


HOLDOUT TEST SET EVALUATION
Sample #001 | Predicted: Sheriff    (99.5%) | Actual: Sheriff    | CORRECT
Sample #002 | Predicted: Rembrandt  (95.4%) | Actual: Rembrandt  | CORRECT
Sample #003 | Predicted: Kvium      (98.7%) | Actual: Kvium      | CORRECT
Sample #004 | Predicted: Sheriff    (99.7%) | Actual: Sheriff    | CORRECT
Sample #005 | Predicted: Rembrandt  (67.7%) | Actual: Rembrandt  | CORRECT
Sample #006 | Predicted: Sheriff    (100.0%) | Actual: Sheriff    | CORRECT
Sample #007 | Predicted: Kvium      (99.3%) | Actual: Kvium      | CORRECT
Sample #008 | Predicted: Rembrandt  (99.8%) | Actual: Rembrandt  | CORRECT
Sample #009 | Predicted: Sheriff    (100.0%) | Actual: Sheriff    | CORRECT
Sample #010 | Predicted: Kvium      (98.3%) | Actual: Kvium      | CORRECT
Sample #011 | Predicted: Heerup     (97.8%) | Actual: Heerup     | CORRECT
Sample #012 | Predicted: Rembrandt  (97.0%) | Actual: Rembrandt  | CORRECT
Sample #013 | Predicted: Heerup     (60.2%) | Actual: Heerup     | CO

In [ ]:
# Print the first 10 spectral bands of the first 3 test samples
print("Sample 1 excerpt:", X_test[0, :10, 0])
print("Sample 2 excerpt:", X_test[1, :10, 0])
print("Sample 3 excerpt:", X_test[2, :10, 0])

# Check if the entire dataset is identical
print("\nIs the data completely frozen?", np.all(X_test[0] == X_test[1]))

In [ ]:
# Check if the loss actually decreased during training
print("Epoch 1 Loss:", history.history['loss'][0])
print("Final Epoch Loss:", history.history['loss'][-1])

print("\nEpoch 1 Accuracy:", history.history['accuracy'][0])
print("Final Epoch Accuracy:", history.history['accuracy'][-1])

## Processign Test data ##


In [ ]:
TEST_CSV_PATH = r"C:\Users\rosha\Downloads\ailearn\HyperLeaf2024\test.csv"          # Kaggle's unlabeled test file
TEST_DATA_DIR = r"C:\Users\rosha\Downloads\ailearn\HyperLeaf2024\images"      # Folder with test .tiff files
MODEL_PATH = "hyperleaf_1d_cnn.keras"
NUM_BANDS = 204

# The original class names (Make sure the order matches how they were one-hot encoded!)
# Typically alphabetical: 0='Heerup', 1='Kvium', 2='Rembrandt', 3='Sheriff'
CLASS_NAMES = ['Heerup', 'Kvium', 'Rembrandt', 'Sheriff']

# ==========================================
# 2. Load the Trained Model
# ==========================================
print("Loading trained model...")
model = tf.keras.models.load_model(MODEL_PATH)

# ==========================================
# 3. Load Test Metadata
# ==========================================
test_df = pd.read_csv(TEST_CSV_PATH)
image_ids = test_df['ImageId'].values

# Initialize empty array for test data
X_test = np.zeros((len(image_ids), NUM_BANDS), dtype=np.float32)

# ==========================================
# 4. Process Test TIFFs (Same logic as Training)
# ==========================================
print("Processing test hyperspectral cubes...")

for i, img_id in enumerate(image_ids):
    file_path = os.path.join(TEST_DATA_DIR, f"{img_id:05d}.tiff")

    # Load TIFF
    cube = tiff.imread(file_path)
    if cube.shape[0] == NUM_BANDS:
        cube = np.transpose(cube, (1, 2, 0))

    # Background Masking
    mean_intensity = np.mean(cube, axis=-1)
    threshold = np.percentile(mean_intensity, 15)
    leaf_mask = mean_intensity > threshold

    # Isolate and Average
    leaf_pixels = cube[leaf_mask]
    if leaf_pixels.shape[0] == 0:
        signature = np.zeros(NUM_BANDS)
    else:
        signature = np.mean(leaf_pixels, axis=0)

    X_test[i] = signature

print(f"Successfully processed {X_test.shape[0]} test samples.")

In [ ]:
X_test = np.expand_dims(X_test, axis=-1)

# ==========================================
# 5. Make Predictions
# ==========================================
print("Running predictions...")
# model.predict outputs probabilities for all 4 classes for every image
predictions_probs = model.predict(X_test)

# Use argmax to find the index of the highest probability (0, 1, 2, or 3)
predicted_indices = np.argmax(predictions_probs, axis=1)

# Map the indices (0-3) back to their text names ('Heerup', etc.)
predicted_cultivars = [CLASS_NAMES[idx] for idx in predicted_indices]